# compressed_tensors — smoke test

Pure Python on top of torch. vLLM uses it to load checkpoints saved by LLM Compressor (quantization schemes, packed 4-bit weights, sparsity bitmasks). The examples quantize a tensor and a small model, pack and unpack weights, and build the Hadamard matrices the package ships data for. Everything runs on the CPU; no GPU and no network are needed.

## Example 1 — the import, the version and the registered compression formats

In [ ]:
import importlib.metadata
import compressed_tensors
from compressed_tensors import BaseCompressor, CompressionFormat

print("compressed_tensors", compressed_tensors.__version__)
assert compressed_tensors.__version__ == importlib.metadata.version("compressed_tensors")
formats = {member.value for member in CompressionFormat}
registered = set(BaseCompressor.registered_names())
assert {"dense", "int-quantized", "float-quantized", "pack-quantized", "naive-quantized"} <= registered, registered
assert registered <= formats, registered - formats  # every registered compressor is a declared format
compressor = BaseCompressor.load_from_registry("pack-quantized")
assert type(compressor).__name__ == "PackedQuantizationCompressor", type(compressor).__name__
print(f"OK -- {len(registered)} compressors registered:", sorted(registered))

## Example 2 — quantize a tensor to 8 bits and back; the error stays within half a step

In [ ]:
import torch
from compressed_tensors.quantization import QuantizationArgs, dequantize, fake_quantize, quantize
from compressed_tensors.quantization.utils import calculate_qparams

torch.manual_seed(0)
weights = torch.randn(16, 32)
args = QuantizationArgs(num_bits=8, symmetric=True)
scale, zero_point = calculate_qparams(weights.amin(), weights.amax(), args)
assert scale.item() > 0 and zero_point.item() == 0, (scale, zero_point)  # symmetric: zero maps to zero
integers = quantize(weights, scale, zero_point, args, dtype=torch.int8)
assert integers.dtype == torch.int8 and -128 <= integers.min() and integers.max() <= 127
restored = dequantize(integers, scale, zero_point, args)
assert torch.equal(restored, fake_quantize(weights, scale, zero_point, args))  # the two paths agree exactly
worst = (restored - weights).abs().max().item()
assert worst <= scale.item() / 2 + 1e-6, (worst, scale.item())
print(f"OK -- 8-bit round trip: step {scale.item():.4f}, worst error {worst:.4f} (at most half a step)")

## Example 3 — attach a quantization scheme to a model's layers

In [ ]:
import torch
from compressed_tensors.quantization import (QuantizationArgs, QuantizationConfig, QuantizationScheme,
                                             QuantizationStatus, apply_quantization_config, preset_name_to_scheme)

model = torch.nn.Sequential(torch.nn.Linear(32, 16), torch.nn.ReLU(), torch.nn.Linear(16, 4))
scheme = QuantizationScheme(targets=["Linear"], weights=QuantizationArgs(num_bits=4, symmetric=True, strategy="channel"))
apply_quantization_config(model, QuantizationConfig(config_groups={"group_0": scheme}))
for layer in (model[0], model[2]):
    assert layer.quantization_status == QuantizationStatus.INITIALIZED, layer.quantization_status
    assert layer.quantization_scheme.weights.num_bits == 4
    assert layer.weight_scale.shape == (layer.out_features, 1), layer.weight_scale.shape  # one scale per channel
    assert layer.weight_zero_point.shape == (layer.out_features, 1)
assert not hasattr(model[1], "quantization_scheme")  # ReLU was not a target
preset = preset_name_to_scheme("W4A16", ["Linear"])  # a named preset: 4-bit weights, 16-bit activations
assert preset.weights.num_bits == 4 and preset.weights.group_size == 128 and preset.input_activations is None
out = model(torch.randn(2, 32))
assert out.shape == (2, 4)
print("OK -- both Linear layers carry a 4-bit per-channel scheme; preset W4A16 =", preset.weights.strategy, "of 128")

## Example 4 — round trips: 4-bit weights packed into 32-bit words, and sparsity bitmasks

In [ ]:
import torch
from compressed_tensors import pack_bitmasks, pack_to_int32, unpack_bitmasks, unpack_from_int32

torch.manual_seed(0)
four_bit = torch.randint(-8, 8, (6, 20), dtype=torch.int8)
packed = pack_to_int32(four_bit, 4)
assert packed.dtype == torch.int32 and packed.shape == (6, 3), (packed.dtype, packed.shape)  # 20 values of 4 bits = 80 bits
assert torch.equal(unpack_from_int32(packed, 4, four_bit.shape), four_bit)
two_bit = torch.randint(-2, 2, (5, 33), dtype=torch.int8)
assert torch.equal(unpack_from_int32(pack_to_int32(two_bit, 2), 2, two_bit.shape), two_bit)
mask = torch.rand(8, 40) > 0.5
bits = pack_bitmasks(mask)
assert bits.dtype == torch.uint8 and bits.shape == (8, 5), (bits.dtype, bits.shape)  # 40 flags = 5 bytes a row
assert torch.equal(unpack_bitmasks(bits, list(mask.shape)), mask)
print(f"OK -- {four_bit.numel()} 4-bit values fit in {packed.numel()} words and come back; so does a {tuple(mask.shape)} bitmask")

## Example 5 — Hadamard matrices, including the sizes that need the data file the package ships

In [ ]:
import os, torch
import compressed_tensors.transform.utils.hadamard as hadamard

data_file = os.path.join(os.path.dirname(hadamard.__file__), "hadamards.safetensors")
assert os.path.isfile(data_file), "the packaged hadamards.safetensors is missing"  # non-Python package data
generator = torch.Generator().manual_seed(0)
for size in (8, 12, 20, 24):  # 12 and 20 are not powers of two: they are built from the packaged matrices
    matrix = hadamard.random_hadamard_matrix(size, dtype=torch.float64, gen=generator)
    assert matrix.abs().unique().tolist() == [1.0], f"entries of size {size} are not all +-1"
    assert torch.allclose(matrix @ matrix.T, size * torch.eye(size, dtype=torch.float64)), size
fixed = hadamard.deterministic_hadamard_matrix(16, dtype=torch.float32)
assert torch.allclose(fixed @ fixed.T, 16 * torch.eye(16))
refused = []
for call in (lambda: hadamard.deterministic_hadamard_matrix(12), lambda: hadamard.random_hadamard_matrix(7)):
    try:
        call()
        raise AssertionError("an impossible Hadamard size was accepted")
    except ValueError as error:
        refused.append(str(error))
print("OK -- sizes 8, 12, 20, 24 are orthogonal with entries +-1; refused:", refused)